## Upper Path (Instrument Trends Analyzer)

**Core Objective:** Map isolated news topics to specific financial instruments to determine localized market movements ("What specifically is moving?").

**Pipeline Steps:**
1. **Mapping:** Evaluates each topic and assigns it to applicable instruments (e.g., AAPL, OIL, BTC) using an LLM with strict JSON output. Topics not affecting known instruments are discarded.
2. **Aggregation:** Groups mapped topics by instrument within a predefined time window (e.g., 15 min, 1 hour).
3. **Severity Calculation:** Computes the signal strength (`severity`) based on topic volume and relevance.
4. **Trend Generation:** Synthesizes grouped topics via LLM to output a consolidated trend direction (`up`, `down`, `neutral`) and a summary of the market impact.

**Output:** `INSTRUMENT_TRENDS` DataFrame containing actionable, asset-specific signals.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

from nlp_predictor.config import Config

# 1. Import the analyzer AND the central Config
from nlp_predictor.instrument_trends import UpperPathAnalyzer

# 2. Resolve the project root from the notebook working directory
notebook_dir = Path.cwd().resolve()
BASE_DIR = notebook_dir.parent

# 3. Ensure the 'src' directory is on sys.path so the local package can be imported
src_dir = BASE_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))


print("====== UpperPathAnalyzer Module Test ======")

try:
    upper_path_analyzer = UpperPathAnalyzer(Config["INSTRUMENT_PATH"])

    # Load data using paths from Config
    df_topics = pd.read_csv(Config["TOPICS_PATH"])
    df_articles = pd.read_csv(Config["ARTICLES_PATH"])
    df_weights = pd.read_csv(Config["WEIGHTS_PATH"])

    df_article_mapping = pd.merge(
        df_articles[["doc_hash", "source_name"]],
        df_weights[["source_name", "source_weight"]],
        on="source_name",
        how="left",
    )

    df_topics_enriched = pd.merge(
        df_topics,
        df_article_mapping[["doc_hash", "source_weight", "source_name"]],
        on="doc_hash",
        how="left",
    )

    df_topics_enriched["source_weight"] = df_topics_enriched["source_weight"].fillna(
        1.0
    )
    df_topics_enriched["date"] = pd.to_datetime(df_topics_enriched["date"], utc=True)

    print("[INFO] Successfully merged source weights into topics.")

    result = upper_path_analyzer.process_data(df_topics_enriched)
    print("====== Processing Upper Path Completed ======")

    print("====== Saving Results To File ======")
    saving_result = UpperPathAnalyzer.save_trends(
        result, Config["INSTRUMENT_TRENDS_SAVE_PATH_PARQUET"]
    )
    if saving_result:
        print("====== Saving Completed ======")
    else:
        print("====== Saving Failed ======")

    # For debugging: save the result to a CSV as well
    result.to_csv(Config["INSTRUMENT_TRENDS_SAVE_PATH_CSV"], index=False)
    print(
        "[DEBUG] Also saved results to CSV for quick inspection at "
        + f"{Config['INSTRUMENT_TRENDS_SAVE_PATH_CSV']}"
    )

except Exception as e:
    print(f"[FATAL ERROR] An error occurred during the UpperPathAnalyzer test: {e}")